# Clase 08 — Contenedores y CI/CD

**Curso:** MLOps y Monitorización con Python · **Bloque B**
**Tiempo estimado:** 110 minutos · **Capítulo del manual:** `manual/08_contenedores_cicd.md`

## Qué verás en esta clase

- Escribir un **`Dockerfile`** real para empaquetar la API de la clase 7, y entender qué problema de reproducibilidad resuelve que fijar versiones con `requirements.txt` (clase 2) NO resuelve.
- Escribir un flujo real de **GitHub Actions** (`ci.yml`) que instala dependencias, ejecuta la suite de tests y, si pasan, "registra" el modelo.
- Validar de verdad, en Python, la sintaxis y la estructura de ambos archivos, sin necesitar Docker ni GitHub instalados.
- Ejecutar localmente, sin Docker, la única parte genuinamente ejecutable: los tests que la CI lanzaría.
- Usar el **registro de modelos de MLflow** (alias y tags) para marcar una versión como candidata, en producción o archivada.
- Entender la diferencia entre **CI** y **CD**, y por qué en ML conviene un CD más cauteloso.

> **Atención (límite de este entorno):** Docker y GitHub Actions no se pueden ejecutar de verdad dentro de Colab ni de este entorno de validación. Este notebook genera y valida el contenido real de ambos archivos, y ejecuta localmente solo la parte que es genuinamente ejecutable sin ellos (los tests). La sección 9 del capítulo explica el comando exacto para probar el flujo completo fuera de aquí.

Todo ocurre dentro de una carpeta de trabajo efímera (`proyecto_churn/`), creada al principio y **eliminada al final** de este mismo notebook.

## Prerequisitos

- De la clase 7: la API FastAPI (`Cliente`, `/predecir`, `/predecir_lote`) que este capítulo empaqueta.
- De la clase 6: la idea de una suite de tests con `pytest` (unitarios, de datos, de modelo).
- De la clase 4: `mlflow.sklearn.log_model` y la idea de un modelo registrado como artefacto versionado.
- De la clase 2: `requirements.txt` con versiones exactas, y por qué eso no basta para fijar el entorno completo.

> **Cómo usar este notebook:** ejecuta las celdas en orden, de arriba a abajo, una sola vez. Tarda menos de un minuto en total.

---
## Instalación

Instala solo lo que falte (incluye FastAPI, httpx, pytest, PyYAML y MLflow).

In [ ]:
import os

# Fijamos estas variables ANTES de la primera vez que se importe mlflow (incluso la comprobación de abajo),
# para que ningún mensaje automático de MLflow dependa de la hora del sistema en la salida de este notebook
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import subprocess
import sys


def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("scikit-learn", "sklearn")
asegurar("joblib", "joblib")
asegurar("fastapi", "fastapi")
asegurar("httpx", "httpx")
asegurar("uvicorn", "uvicorn")
asegurar("pytest", "pytest")
asegurar("PyYAML", "yaml")
asegurar("mlflow", "mlflow")
print("Librerías listas.")

---
## Celda 0 — Setup

In [ ]:
import os
import re
import shutil
import subprocess
import sys
import json
import logging
import warnings

import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from importlib.metadata import version
import yaml

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

# Igual que en la clase 4: fijamos estas variables ANTES de importar mlflow y subimos su logger a ERROR
# (WARNING no basta: mlflow.utils.requirements_utils avisa en WARNING con detalles no deterministicos)
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import mlflow
import mlflow.sklearn
from mlflow import MlflowClient

logging.getLogger("mlflow").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=DeprecationWarning)

SEMILLA = 42
COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]
COLUMNAS_ENTRADA = COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS


def ejecutar_pytest(argumentos):
    """Ejecuta pytest con subprocess.run y devuelve un resumen SIN los tiempos de ejecución (cambian en cada ejecución)."""
    resultado = subprocess.run([sys.executable, "-m", "pytest", "-q"] + argumentos, capture_output=True, text=True)
    salida = resultado.stdout + resultado.stderr
    m_passed = re.search(r"(\d+) passed", salida)
    m_failed = re.search(r"(\d+) failed", salida)
    m_error = re.search(r"(\d+) error", salida)
    return {
        "returncode": resultado.returncode,
        "passed": int(m_passed.group(1)) if m_passed else 0,
        "failed": int(m_failed.group(1)) if m_failed else 0,
        "error": int(m_error.group(1)) if m_error else 0,
    }


print("Librerías y funciones auxiliares listas. Versión de MLflow:", mlflow.__version__)

---
## Carpeta de trabajo efímera

In [ ]:
CARPETA = "proyecto_churn"

# Limpiamos y recreamos la carpeta para que el notebook sea repetible desde cero
if os.path.exists(CARPETA):
    shutil.rmtree(CARPETA)
os.makedirs(os.path.join(CARPETA, "tests"))
print("Carpeta de trabajo creada:", os.path.abspath(CARPETA))

---
## 1. Reentrenar y guardar el artefacto (como en la clase 7)

La API que vamos a empaquetar necesita un `modelo.pkl` real en disco.

In [ ]:
RUTA_CSV = cargar_datos("churn.csv")
df = pd.read_csv(RUTA_CSV)

X = df[COLUMNAS_ENTRADA]
y = df["abandono"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEMILLA, stratify=y
)

# La misma configuración ganadora de las clases 4, 5 y 7: regresión logística, class_weight="balanced", C=1.0
preprocesador = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), COLUMNAS_CATEGORICAS),
    ("numericas", StandardScaler(), COLUMNAS_NUMERICAS),
])
clasificador = LogisticRegression(class_weight="balanced", random_state=SEMILLA, max_iter=1000, C=1.0)
pipeline = Pipeline([("preprocesado", preprocesador), ("clasificador", clasificador)])
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]
F1 = round(f1_score(y_test, y_pred), 4)
AUC = round(roc_auc_score(y_test, y_proba), 4)
print("F1 (conjunto de prueba):", F1)
print("AUC (conjunto de prueba):", AUC)
assert F1 > 0.25 and AUC > 0.6, "el reentrenamiento no reprodujo el modelo esperado"

RUTA_MODELO = os.path.join(CARPETA, "modelo.pkl")
joblib.dump(pipeline, RUTA_MODELO)
print("Artefacto guardado en:", RUTA_MODELO)

---
## 2. `requirements.txt`: necesario, pero no suficiente

Como en la clase 2, generamos `requirements.txt` con las versiones **realmente instaladas**, con `importlib.metadata.version`. La diferencia es el alcance: aquí solo van las librerías que la API necesita en TIEMPO DE EJECUCIÓN dentro del contenedor.

In [ ]:
# Igual que en la clase 2 (importlib.metadata.version, nunca escritas a mano), pero ahora las 6 librerías
# que la API de la clase 7 necesita en TIEMPO DE EJECUCIÓN dentro del contenedor: no incluye pytest ni mlflow,
# que son herramientas de desarrollo/CI, no dependencias del servicio que se despliega
PAQUETES_API = [
    ("numpy", "numpy"), ("pandas", "pandas"), ("scikit-learn", "scikit-learn"),
    ("joblib", "joblib"), ("fastapi", "fastapi"), ("uvicorn", "uvicorn"),
]
versiones_api = {nombre_pip: version(nombre_pip) for nombre_pip, _ in PAQUETES_API}

RUTA_REQUIREMENTS = os.path.join(CARPETA, "requirements.txt")
with open(RUTA_REQUIREMENTS, "w") as f:
    f.write("\n".join(f"{paquete}=={ver}" for paquete, ver in versiones_api.items()) + "\n")

print("requirements.txt generado:")
print(open(RUTA_REQUIREMENTS).read())

> **Atención:** `requirements.txt` fija QUÉ paquetes de Python y en qué versión, pero no fija: la versión de **Python** misma, el **sistema operativo** (¿Debian? ¿Alpine? ¿Windows?), las **librerías C del sistema** de las que dependen paquetes como `numpy` o `scikit-learn` por debajo (BLAS/LAPACK), ni la **arquitectura del procesador** (x86_64 frente a ARM). Dos máquinas con el mismo `requirements.txt` instalado pueden seguir comportándose distinto si difieren en alguna de esas capas -- exactamente el problema que la clase 2 dejó anunciado y que resuelve la sección 4.

---
## 3. Empaquetar la API de la clase 7 en un archivo real

Escribimos `api.py`: la misma `Cliente`, el mismo `/predecir` y `/predecir_lote` de la clase 7, pero como un módulo independiente que carga `modelo.pkl` al arrancar -- exactamente lo que un contenedor Docker ejecutaría con `uvicorn`.

In [ ]:
API_PY_CODE = '\n"""API de churn (clase 7), empaquetada para servirse con uvicorn dentro de un contenedor."""\nimport joblib\nimport pandas as pd\nfrom fastapi import FastAPI\nfrom pydantic import BaseModel, Field\nfrom typing import Literal, List\n\nCOLUMNAS_ENTRADA = ["plan", "contrato", "metodo_pago", "antiguedad_meses",\n                     "uso_mensual_horas", "tickets_soporte", "factura_mensual"]\n\nMODELO = joblib.load("modelo.pkl")\n\n\nclass Cliente(BaseModel):\n    id_cliente: str\n    antiguedad_meses: int = Field(ge=0)\n    plan: Literal["basico", "estandar", "premium"]\n    contrato: Literal["mensual", "anual"]\n    uso_mensual_horas: float = Field(ge=0)\n    tickets_soporte: int = Field(ge=0)\n    metodo_pago: Literal["domiciliacion", "tarjeta", "transferencia"]\n    factura_mensual: float = Field(ge=0)\n\n\ndef cliente_a_fila(cliente):\n    return pd.DataFrame([cliente.model_dump()])[COLUMNAS_ENTRADA]\n\n\napp = FastAPI(title="churn-api")\n\n\n@app.post("/predecir")\ndef predecir(cliente: Cliente):\n    fila = cliente_a_fila(cliente)\n    probabilidad = float(MODELO.predict_proba(fila)[0, 1])\n    return {\n        "id_cliente": cliente.id_cliente,\n        "probabilidad_abandono": round(probabilidad, 4),\n        "abandono_previsto": int(probabilidad >= 0.5),\n    }\n\n\n@app.post("/predecir_lote")\ndef predecir_lote(clientes: List[Cliente]):\n    filas = pd.DataFrame([c.model_dump() for c in clientes])[COLUMNAS_ENTRADA]\n    probabilidades = MODELO.predict_proba(filas)[:, 1]\n    return [\n        {"id_cliente": c.id_cliente, "probabilidad_abandono": round(float(p), 4), "abandono_previsto": int(p >= 0.5)}\n        for c, p in zip(clientes, probabilidades)\n    ]\n'

In [ ]:
RUTA_API = os.path.join(CARPETA, "api.py")
with open(RUTA_API, "w") as f:
    f.write(API_PY_CODE.strip() + "\n")
print("api.py escrito en:", RUTA_API, f"({len(API_PY_CODE.splitlines())} líneas)")

---
## 4. Escribir el `Dockerfile` real

Un `Dockerfile` describe, instrucción por instrucción, cómo construir una **imagen**: un sistema de archivos completo (sistema operativo base, Python, librerías del sistema, dependencias de Python y nuestro código), listo para ejecutarse igual en cualquier máquina con Docker instalado.

In [ ]:
DOCKERFILE_CODE = 'FROM python:3.12-slim\n\nWORKDIR /app\n\nCOPY requirements.txt .\nRUN pip install --no-cache-dir -r requirements.txt\n\nCOPY api.py .\nCOPY modelo.pkl .\n\nEXPOSE 8000\n\nCMD ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"]\n'

In [ ]:
RUTA_DOCKERFILE = os.path.join(CARPETA, "Dockerfile")
with open(RUTA_DOCKERFILE, "w") as f:
    f.write(DOCKERFILE_CODE)
print("Dockerfile escrito en:", RUTA_DOCKERFILE)
print(DOCKERFILE_CODE)

| Instrucción | Qué hace aquí |
|---|---|
| `FROM python:3.12-slim` | fija el sistema operativo base Y la versión de Python -- lo que `requirements.txt` NUNCA fija |
| `WORKDIR /app` | crea y entra en `/app` dentro de la imagen |
| `COPY requirements.txt .` + `RUN pip install ...` | instala las dependencias DENTRO de la imagen, con las mismas librerías C del sistema Debian del `FROM` |
| `COPY api.py .` / `COPY modelo.pkl .` | mete nuestro código y el artefacto entrenado en la imagen |
| `EXPOSE 8000` | documenta qué puerto escuchará el contenedor |
| `CMD [...]` | el comando que se ejecuta al arrancar un contenedor de esta imagen |

### 4.1 Validar la sintaxis y la estructura del Dockerfile, en Python, sin Docker

In [ ]:
INSTRUCCIONES_VALIDAS = {
    "FROM", "WORKDIR", "COPY", "ADD", "RUN", "ENV", "EXPOSE", "CMD",
    "ENTRYPOINT", "ARG", "LABEL", "USER", "VOLUME", "STOPSIGNAL", "HEALTHCHECK", "SHELL",
}
# MAINTAINER está oficialmente obsoleto desde Docker 1.13 (usar LABEL en su lugar): lo tratamos como prohibido
INSTRUCCIONES_PROHIBIDAS = {"MAINTAINER"}


def validar_dockerfile(ruta):
    """Valida estructura y orden del Dockerfile leyendo línea por línea, sin usar Docker."""
    instrucciones = []
    with open(ruta) as f:
        for linea in f:
            linea = linea.strip()
            if not linea or linea.startswith("#"):
                continue
            palabra = linea.split()[0].upper()
            instrucciones.append(palabra)

    errores = []
    if not instrucciones or instrucciones[0] != "FROM":
        errores.append("la primera instrucción debe ser FROM")
    prohibidas_usadas = [i for i in instrucciones if i in INSTRUCCIONES_PROHIBIDAS]
    if prohibidas_usadas:
        errores.append(f"instrucciones prohibidas encontradas: {prohibidas_usadas}")
    desconocidas = [i for i in instrucciones if i not in INSTRUCCIONES_VALIDAS and i not in INSTRUCCIONES_PROHIBIDAS]
    if desconocidas:
        errores.append(f"instrucciones desconocidas: {desconocidas}")
    if "COPY" not in instrucciones and "ADD" not in instrucciones:
        errores.append("falta al menos un COPY/ADD para meter código en la imagen")
    if "RUN" not in instrucciones:
        errores.append("falta al menos un RUN (aquí, instalar dependencias)")
    if not instrucciones or instrucciones[-1] not in ("CMD", "ENTRYPOINT"):
        errores.append("la última instrucción debe ser CMD o ENTRYPOINT")
    idx_from = instrucciones.index("FROM") if "FROM" in instrucciones else -1
    idx_copy = next((i for i, x in enumerate(instrucciones) if x in ("COPY", "ADD")), len(instrucciones))
    idx_run = instrucciones.index("RUN") if "RUN" in instrucciones else len(instrucciones)
    if not (idx_from < idx_copy and idx_from < idx_run):
        errores.append("FROM debe preceder a COPY/ADD y a RUN")
    return instrucciones, errores


instrucciones_encontradas, errores_dockerfile = validar_dockerfile(RUTA_DOCKERFILE)
print("instrucciones, en orden:", instrucciones_encontradas)
print("errores encontrados:", errores_dockerfile)
assert errores_dockerfile == [], "el Dockerfile no pasó la validación de estructura"
print("Dockerfile válido: FROM primero, COPY y RUN presentes, CMD al final, sin instrucciones prohibidas.")

**¿Qué observas?** La validación no ejecuta nada dentro de una imagen: lee el archivo línea por línea, ignora comentarios y líneas vacías, y comprueba reglas concretas (FROM primero, al menos un COPY y un RUN, CMD/ENTRYPOINT al final, ninguna instrucción prohibida como `MAINTAINER`). Es una comprobación de **estructura**, no de que la imagen construya correctamente -- eso solo lo confirma `docker build` de verdad (sección 9).

---
## 5. El flujo de GitHub Actions: `ci.yml`

**CI (integración continua)** significa: cada vez que alguien cambia el código, un sistema automático instala las dependencias y ejecuta la suite de tests, sin intervención humana. GitHub Actions es el sistema de CI integrado en GitHub; un flujo (*workflow*) se describe en un archivo YAML dentro de `.github/workflows/`.

In [ ]:
CI_YML_CODE = 'name: CI - churn API\n\non:\n  push:\n    branches: [main]\n  pull_request:\n    branches: [main]\n\njobs:\n  test-and-register:\n    runs-on: ubuntu-latest\n    steps:\n      - name: Descargar el código del repositorio\n        uses: actions/checkout@v4\n\n      - name: Configurar Python\n        uses: actions/setup-python@v5\n        with:\n          python-version: "3.12"\n\n      - name: Instalar dependencias (API + tests)\n        run: pip install -r requirements.txt pytest httpx\n\n      - name: Ejecutar la suite de tests (clase 6)\n        run: pytest tests/ -q\n\n      - name: Registrar el modelo (paso simbolico, clase 4)\n        if: success()\n        run: echo "Tests superados -- registrando el modelo como candidato en el registro de MLflow"\n'

In [ ]:
os.makedirs(os.path.join(CARPETA, ".github", "workflows"), exist_ok=True)
RUTA_CI_YML = os.path.join(CARPETA, ".github", "workflows", "ci.yml")
with open(RUTA_CI_YML, "w") as f:
    f.write(CI_YML_CODE)
print("ci.yml escrito en:", RUTA_CI_YML)
print(CI_YML_CODE)

| Sección del YAML | Qué significa |
|---|---|
| `name:` | el nombre que verás en la pestaña "Actions" de GitHub |
| `on:` | los eventos que disparan el flujo (aquí: `push`/`pull_request` a `main`) |
| `jobs:` | uno o más trabajos, cada uno con su propia máquina virtual |
| `runs-on: ubuntu-latest` | qué sistema operativo usa la máquina virtual del job |
| `steps:` | la lista ordenada de pasos dentro del job |
| `uses: actions/checkout@v4` | un paso reutilizado de otro repositorio (aquí, descargar el código) |
| `run: pytest tests/ -q` | un paso que ejecuta un comando de shell directamente |
| `if: success()` | condición: este paso solo se ejecuta si TODOS los pasos anteriores tuvieron éxito |

### 5.1 Validar la sintaxis y la estructura del ci.yml, en Python, con `yaml.safe_load`

In [ ]:
with open(RUTA_CI_YML) as f:
    flujo = yaml.safe_load(f.read())

# Atención: en YAML 1.1 (el que usa PyYAML) la clave "on" sin comillas se interpreta como el booleano True,
# no como la cadena "on" -- por eso comprobamos las dos formas posibles de la clave
clave_on = "on" if "on" in flujo else (True if True in flujo else None)

errores_yml = []
if "name" not in flujo:
    errores_yml.append("falta la clave 'name'")
if clave_on is None:
    errores_yml.append("falta la clave 'on' (disparadores)")
if "jobs" not in flujo or not flujo["jobs"]:
    errores_yml.append("falta la clave 'jobs' o está vacía")
else:
    for nombre_job, job in flujo["jobs"].items():
        if "runs-on" not in job:
            errores_yml.append(f"el job '{nombre_job}' no tiene 'runs-on'")
        if "steps" not in job or not job["steps"]:
            errores_yml.append(f"el job '{nombre_job}' no tiene 'steps'")

print("claves de primer nivel:", sorted(str(k) for k in flujo.keys()))
print("n. de steps del job:", len(flujo["jobs"]["test-and-register"]["steps"]))
print("errores encontrados:", errores_yml)
assert errores_yml == [], "el ci.yml no pasó la validación de estructura"
print("ci.yml válido: name, on, jobs, runs-on y steps presentes.")

**¿Qué observas?** `yaml.safe_load` hace un parseo REAL del archivo: si el YAML estuviera mal formado (indentación incorrecta, dos puntos donde no debe haberlos), esta celda lanzaría una excepción antes de llegar al `assert`. La comprobación de las claves esperadas (`name`, `on`, `jobs`, `runs-on`, `steps`) confirma además que el archivo tiene la ESTRUCTURA que GitHub Actions espera, no solo que es YAML válido en general.

> **Atención (trampa real de YAML):** en YAML 1.1 -- el que usa PyYAML por defecto -- una clave `on:` sin comillas se interpreta como el booleano `True`, no como la cadena `"on"`. Si buscaras `"on" in flujo` sin más, la comprobación fallaría en silencio aunque el archivo esté perfectamente bien escrito. Es un error real y documentado de la comunidad de GitHub Actions con herramientas basadas en PyYAML, no una curiosidad inventada para este capítulo.

---
## 6. Escribir los tests que la CI ejecutaría (clase 6 + clase 7)

Escribimos `tests/test_modelo.py` (como en la clase 6: el F1 no debe caer bajo un umbral) y `tests/test_api.py` (como en la clase 5 -- perdón, clase 7 -- con `TestClient`, sobre `api.py` de verdad).

In [ ]:
TEST_MODELO_PY = '\nimport joblib\nimport pandas as pd\nfrom sklearn.metrics import f1_score\n\nF1_MINIMO = 0.28\n\n\ndef test_modelo_supera_el_umbral_de_f1():\n    """Como en la clase 6: el modelo actual no debe empeorar por debajo de un umbral conocido."""\n    modelo = joblib.load("modelo.pkl")\n    df = pd.read_csv(RUTA_CSV_TESTS)\n    columnas = ["plan", "contrato", "metodo_pago", "antiguedad_meses",\n                "uso_mensual_horas", "tickets_soporte", "factura_mensual"]\n    predicciones = modelo.predict(df[columnas])\n    f1 = f1_score(df["abandono"], predicciones)\n    assert f1 >= F1_MINIMO\n'
TEST_API_PY = '\nfrom fastapi.testclient import TestClient\nfrom api import app\n\ncliente_api = TestClient(app)\n\nCLIENTE_VALIDO = {\n    "id_cliente": "C_TEST_0001", "antiguedad_meses": 12, "plan": "estandar", "contrato": "mensual",\n    "uso_mensual_horas": 15.0, "tickets_soporte": 1, "metodo_pago": "tarjeta", "factura_mensual": 22.5,\n}\n\n\ndef test_predecir_con_cliente_valido_responde_200():\n    respuesta = cliente_api.post("/predecir", json=CLIENTE_VALIDO)\n    assert respuesta.status_code == 200\n    assert set(respuesta.json().keys()) == {"id_cliente", "probabilidad_abandono", "abandono_previsto"}\n\n\ndef test_predecir_con_plan_invalido_responde_422():\n    cliente_malo = dict(CLIENTE_VALIDO, plan="diamante")\n    respuesta = cliente_api.post("/predecir", json=cliente_malo)\n    assert respuesta.status_code == 422\n'

In [ ]:
# test_modelo.py necesita la ruta absoluta al CSV porque pytest se ejecuta con proyecto_churn/ como directorio actual
ruta_csv_absoluta = os.path.abspath(RUTA_CSV)
texto_test_modelo = "RUTA_CSV_TESTS = " + repr(ruta_csv_absoluta) + "\n" + TEST_MODELO_PY

with open(os.path.join(CARPETA, "tests", "test_modelo.py"), "w") as f:
    f.write(texto_test_modelo)
with open(os.path.join(CARPETA, "tests", "test_api.py"), "w") as f:
    f.write(TEST_API_PY)
with open(os.path.join(CARPETA, "tests", "__init__.py"), "w") as f:
    f.write("")
print("tests escritos en:", os.path.join(CARPETA, "tests"))

### 6.1 Ejecutar la suite LOCALMENTE (lo único genuinamente ejecutable sin Docker/GitHub aquí)

In [ ]:
# Esto es EXACTAMENTE lo que la CI ejecutaría en el paso "Ejecutar la suite de tests (clase 6)":
# pytest sobre la carpeta tests/, con conftest capaz de importar api.py como módulo (sys.path incluye CARPETA)
sys.path.insert(0, os.path.abspath(CARPETA))
resultado_pytest = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests"],
    capture_output=True, text=True, cwd=CARPETA,
)
salida_pytest = resultado_pytest.stdout + resultado_pytest.stderr
m_passed = re.search(r"(\d+) passed", salida_pytest)
m_failed = re.search(r"(\d+) failed", salida_pytest)
resumen_tests = {
    "returncode": resultado_pytest.returncode,
    "passed": int(m_passed.group(1)) if m_passed else 0,
    "failed": int(m_failed.group(1)) if m_failed else 0,
}
print("resumen de la suite (sin tiempos, no son deterministas):", resumen_tests)
assert resumen_tests["failed"] == 0 and resumen_tests["returncode"] == 0
print("Los 3 tests pasan: 1 del modelo (clase 6) y 2 de la API (clase 7).")

**¿Qué observas?** Este `subprocess.run(["pytest", ...])` es literalmente el mismo comando que el paso `run: pytest tests/ -q` de `ci.yml` ejecutaría en una máquina virtual de GitHub. La diferencia es el entorno: aquí corre en el mismo proceso Python de este notebook, no en un contenedor limpio -- por eso la sección 9 declara explícitamente que esto NO sustituye a ejecutar el flujo completo en un repositorio real.

In [ ]:
# El mismo "if: success()" que en ci.yml, ahora en Python: solo seguimos si la suite pasó
if resumen_tests["failed"] == 0:
    print("Tests superados -- registrando el modelo como candidato (sección siguiente).")
else:
    raise SystemExit("tests fallidos: NO se registra el modelo")

---
## 7. El registro de modelos de MLflow: candidato, producción, archivado

Como en la clase 4, el almacén de MLflow es una carpeta LOCAL dentro de `proyecto_churn/`. El **registro de modelos** añade sobre eso un concepto: un modelo con NOMBRE (`churn_modelo`), con varias VERSIONES numeradas, cada una apuntando a un run concreto. Un **alias** (`candidato`, `produccion`, `archivado`) apunta a una única versión y se puede reasignar -- es el mecanismo actual de MLflow (sustituye a los antiguos *stages*, ya obsoletos).

In [ ]:
RUTA_MLRUNS = os.path.join(CARPETA, "mlruns")
mlflow.set_tracking_uri("file:" + RUTA_MLRUNS)
NOMBRE_EXPERIMENTO = "churn_registro"
mlflow.set_experiment(NOMBRE_EXPERIMENTO)
NOMBRE_MODELO_REGISTRADO = "churn_modelo"

print("Tracking URI:", mlflow.get_tracking_uri())
assert mlflow.get_tracking_uri().startswith("file:") and "proyecto_churn" in mlflow.get_tracking_uri()

### 7.1 Registrar la versión 1 (ya "en producción") y la versión 2 (candidata)

In [ ]:
# Versión 1: la configuración SIN balancear (la que "ya estaba en producción" antes de esta clase, clase 4)
clasificador_v1 = LogisticRegression(random_state=SEMILLA, max_iter=1000)
pipeline_v1 = Pipeline([("preprocesado", preprocesador), ("clasificador", clasificador_v1)])
pipeline_v1.fit(X_train, y_train)
f1_v1 = round(f1_score(y_test, pipeline_v1.predict(X_test)), 4)
print("F1 de la versión 1 (sin balancear):", f1_v1)

with mlflow.start_run(run_name="version_1_sin_balancear") as run_v1:
    mlflow.log_param("class_weight", "ninguno")
    mlflow.log_metric("f1", f1_v1)
    mlflow.sklearn.log_model(pipeline_v1, name="modelo", pip_requirements=[f"scikit-learn=={sklearn.__version__}"])
    id_run_v1 = run_v1.info.run_id

version_v1 = mlflow.register_model(f"runs:/{id_run_v1}/modelo", NOMBRE_MODELO_REGISTRADO)
print("versión registrada:", version_v1.version)

In [ ]:
# Versión 2: la configuración ganadora (balanceada) que este notebook reentrenó al principio
with mlflow.start_run(run_name="version_2_balanceada") as run_v2:
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("f1", F1)
    mlflow.sklearn.log_model(pipeline, name="modelo", pip_requirements=[f"scikit-learn=={sklearn.__version__}"])
    id_run_v2 = run_v2.info.run_id

version_v2 = mlflow.register_model(f"runs:/{id_run_v2}/modelo", NOMBRE_MODELO_REGISTRADO)
print("versión registrada:", version_v2.version)

In [ ]:
cliente_mlflow = MlflowClient()

# Un "alias" apunta a UNA versión concreta y se puede reasignar: es el mecanismo que sustituye a los antiguos "stages"
cliente_mlflow.set_registered_model_alias(NOMBRE_MODELO_REGISTRADO, "produccion", version_v1.version)
cliente_mlflow.set_registered_model_alias(NOMBRE_MODELO_REGISTRADO, "candidato", version_v2.version)
cliente_mlflow.set_model_version_tag(NOMBRE_MODELO_REGISTRADO, version_v2.version, "f1_test", str(F1))

version_en_produccion = cliente_mlflow.get_model_version_by_alias(NOMBRE_MODELO_REGISTRADO, "produccion").version
version_candidata = cliente_mlflow.get_model_version_by_alias(NOMBRE_MODELO_REGISTRADO, "candidato").version
print("alias 'produccion' -> versión:", version_en_produccion)
print("alias 'candidato' -> versión:", version_candidata)
assert version_en_produccion == version_v1.version and version_candidata == version_v2.version

**¿Qué observas?** Dos versiones del MISMO modelo registrado (`churn_modelo`), cada una con su alias: la 1 en `produccion` (la que hipotéticamente ya estaba sirviendo tráfico), la 2 en `candidato` (recién entrenada, a la espera de validación). Ningún cliente real recibe todavía predicciones de la versión 2 -- eso es exactamente lo que separa CI de CD (sección 9).

### 7.2 Promover el candidato a producción (solo si la CI pasó Y no empeora la métrica)

In [ ]:
# El paso "cauteloso" (sección 9 del capítulo): el candidato SOLO se promueve porque los tests de la CI ya pasaron
# (resumen_tests["failed"] == 0, comprobado más arriba) Y porque su F1 no es peor que el de la versión en producción
f1_produccion_actual = float(cliente_mlflow.get_model_version(NOMBRE_MODELO_REGISTRADO, version_en_produccion).tags.get("f1_test", f1_v1))
puede_promover = resumen_tests["failed"] == 0 and F1 >= f1_produccion_actual - 1e-9
print("¿se puede promover el candidato a producción?", puede_promover, f"(F1 candidato={F1}, F1 producción actual={f1_produccion_actual})")
assert puede_promover

# Reasignar el alias "produccion" a la v2 lo QUITA automáticamente de la v1 (un alias apunta a una sola versión)
cliente_mlflow.set_registered_model_alias(NOMBRE_MODELO_REGISTRADO, "produccion", version_v2.version)
cliente_mlflow.delete_registered_model_alias(NOMBRE_MODELO_REGISTRADO, "candidato")
cliente_mlflow.set_registered_model_alias(NOMBRE_MODELO_REGISTRADO, "archivado", version_v1.version)
cliente_mlflow.set_model_version_tag(NOMBRE_MODELO_REGISTRADO, version_v1.version, "estado", "archivado")

version_en_produccion_final = cliente_mlflow.get_model_version_by_alias(NOMBRE_MODELO_REGISTRADO, "produccion").version
version_archivada = cliente_mlflow.get_model_version_by_alias(NOMBRE_MODELO_REGISTRADO, "archivado").version
print("alias 'produccion' ahora apunta a la versión:", version_en_produccion_final)
print("alias 'archivado' apunta a la versión:", version_archivada)
assert version_en_produccion_final == version_v2.version and version_archivada == version_v1.version
print("Ciclo de vida completo: v1 candidato->producción->archivado, v2 candidato->producción.")

**¿Qué observas?** La promoción no es automática "porque los tests pasaron": comprueba DOS condiciones (`resumen_tests["failed"] == 0` Y `F1 >= f1_produccion_actual`) antes de mover el alias `produccion`. Reasignar un alias lo QUITA automáticamente de la versión anterior -- por eso, tras promover la v2, comprobamos que `produccion` ya no apunta a la v1, y le asignamos el alias `archivado` en su lugar.

---
## 8. CI frente a CD: por qué en ML conviene un CD más cauteloso

**CI (integración continua)**: cada cambio de código dispara automáticamente instalación + tests. Si algo falla, se sabe en minutos, no en la siguiente vez que alguien mire el proyecto. Eso es exactamente lo que hace `ci.yml` (secciones 5 y 6).

**CD (despliegue continuo)**: si los tests pasan, el cambio se publica automáticamente -- en software tradicional, a menudo sin ningún paso manual entre "tests verdes" y "en producción".

En ML, promover un modelo con los tests en verde NO es suficiente, por razones que no existen en software tradicional:

| Riesgo específico de ML | Por qué un test unitario no lo detecta |
|---|---|
| **Drift**: la distribución de los datos reales cambia con el tiempo (clase 11) | los tests de la sección 6 comprueban el CSV de hoy, no los datos de dentro de tres meses |
| Una métrica global aceptable puede esconder un subgrupo mal servido | `f1_score` sobre todo el conjunto de prueba no dice nada sobre un segmento de clientes concreto |
| El "correcto" cambia con el negocio, no solo con el código | un modelo puede seguir siendo técnicamente correcto y ya no ser la decisión de negocio adecuada |

Por eso, un CD cauteloso en ML suele añadir, ENTRE "tests verdes" y "sirviendo el 100% del tráfico": una promoción explícita del alias (como la sección 7.2, no automática por completo), un despliegue **canary** o **shadow** (clase 9) que compara el candidato con la producción actual antes de sustituirla del todo, y una revisión humana de la métrica antes de mover el alias `produccion`. La sección 7.2 de este notebook es exactamente ese paso intermedio: automatizado, pero con una condición explícita, no un `if: success()` desnudo.

---
## 9. Probar el flujo completo fuera de este notebook

> **Atención:** nada de lo siguiente se ejecuta en este notebook. Son los comandos EXACTOS que lanzarías en tu propia máquina o repositorio.
>
> **Con Docker instalado en local**, dentro de `proyecto_churn/` (o la carpeta donde tengas `Dockerfile`, `api.py`, `modelo.pkl` y `requirements.txt`):
> ```
> docker build -t churn-api:1.0 .
> docker run -p 8000:8000 churn-api:1.0
> ```
> y en otra terminal:
> ```
> curl -X POST http://localhost:8000/predecir -H "Content-Type: application/json" \
>   -d "{\"id_cliente\":\"C1\",\"antiguedad_meses\":12,\"plan\":\"estandar\",\"contrato\":\"mensual\",\"uso_mensual_horas\":15.0,\"tickets_soporte\":1,\"metodo_pago\":\"tarjeta\",\"factura_mensual\":22.5}"
> ```
>
> **Con un repositorio propio en GitHub**: copia `.github/workflows/ci.yml` (y `requirements.txt`, `api.py`, `modelo.pkl`, `tests/`) a un repositorio, haz `git push`, y abre la pestaña **Actions** del repositorio en github.com. Verás el job `test-and-register` ejecutándose paso a paso, en vivo, con el resultado real de `pytest` -- exactamente los mismos pasos que validamos en las secciones 5 y 6, pero corriendo de verdad en la infraestructura de GitHub.

### 9.1 ¿Hay Docker instalado en este entorno?

In [ ]:
resultado_which = subprocess.run(["which", "docker"], capture_output=True, text=True)
docker_disponible = resultado_which.returncode == 0
print("¿hay un ejecutable 'docker' en este entorno?:", docker_disponible)
if docker_disponible:
    print("(ruta encontrada, pero NO lo usamos en este notebook -- ver el recuadro de Atención del capítulo)")
else:
    print("(no encontrado en este entorno -- es lo esperado en Colab/este entorno de validación, ver el recuadro de Atención)")

---
## 10. Limpieza final de la carpeta de trabajo

In [ ]:
shutil.rmtree(CARPETA)
print("Carpeta de trabajo eliminada:", CARPETA)
assert not os.path.exists(CARPETA)

---
## Resumen

- Un `Dockerfile` real (`FROM`/`WORKDIR`/`COPY`/`RUN`/`EXPOSE`/`CMD`) empaqueta la API de la clase 7 con su sistema operativo base, Python y librerías C -- lo que `requirements.txt` (clase 2) nunca fija.
- Un `ci.yml` real de GitHub Actions instala dependencias, ejecuta `pytest` y, si pasa, "registra" el modelo (paso simbólico, `if: success()`).
- Ambos archivos se VALIDARON de verdad en Python (estructura línea a línea del Dockerfile; `yaml.safe_load` + claves esperadas del `.yml`), sin necesitar Docker ni GitHub instalados.
- Los tests que la CI ejecutaría se corrieron LOCALMENTE con `subprocess.run(["pytest", ...])`: es la única parte genuinamente ejecutable en este entorno.
- El **registro de modelos de MLflow** (alias `candidato`/`produccion`/`archivado`, reasignables) marcó el ciclo de vida completo de dos versiones del mismo modelo.
- **CI** prueba automáticamente cada cambio; **CD** lo publica automáticamente si pasa. En ML conviene un CD más cauteloso: drift, subgrupos mal servidos y decisiones de negocio son riesgos que un test en verde no detecta por sí solo.

*Continúa con la clase 9: Despliegue y publicación.*